# 01. 같은 구 내 이동 상세 분석

탑승완료 데이터를 기준으로 `출발구 == 목적구`인 서울 자치구 내 이동을 상세히 분석한다.
구별 이동 건수, 같은 구 내 이동 비율, 승차거리, 대기시간을 함께 확인하여 구 내 이동 수요가 큰 지역과 그 특성을 파악한다.


## 1. 데이터 로드

정제된 탑승내역 데이터를 불러온다. 노트북 실행 위치가 프로젝트 루트이거나 `notebooks_kms/` 내부인 경우 모두 동작하도록 프로젝트 루트와 데이터 경로를 설정한다.


In [ ]:
import platform
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists() and (PROJECT_ROOT.parent / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"

if platform.system() == "Darwin":
    plt.rcParams["font.family"] = "AppleGothic"
elif platform.system() == "Windows":
    plt.rcParams["font.family"] = "Malgun Gothic"
else:
    plt.rcParams["font.family"] = "NanumGothic"

plt.rcParams["axes.unicode_minus"] = False

boarding_path = PROCESSED_DIR / "서울시설공단_장애인콜택시 탑승내역_정제_20251231.csv"
print(boarding_path)

df = pd.read_csv(boarding_path)
print(f"전체 행 수: {len(df):,}")
display(df.head())


## 2. 탑승완료 필터링

이동 흐름 분석은 실제 승차와 하차가 완료된 건을 기준으로 한다. `승차일시`와 `하차일시`가 있고 `취소일시`가 없는 행만 탑승완료 데이터로 분리한다.


In [ ]:
completed = df[
    df["승차일시"].notna()
    & df["하차일시"].notna()
    & df["취소일시"].isna()
].copy()

print(f"탑승완료 건수: {len(completed):,} / 전체 {len(df):,} ({len(completed) / len(df) * 100:.1f}%)")
display(completed.head())


## 3. 서울 25개구 필터링

서울 자치구 내 이동만 분석하기 위해 출발구와 목적구가 모두 서울 25개 자치구에 포함되는 탑승완료 건을 분리한다.


In [ ]:
SEOUL_25 = [
    "강남구", "강동구", "강북구", "강서구", "관악구",
    "광진구", "구로구", "금천구", "노원구", "도봉구",
    "동대문구", "동작구", "마포구", "서대문구", "서초구",
    "성동구", "성북구", "송파구", "양천구", "영등포구",
    "용산구", "은평구", "종로구", "중구", "중랑구",
]

seoul_completed = completed[
    completed["출발구"].isin(SEOUL_25)
    & completed["목적구"].isin(SEOUL_25)
].copy()

print(f"서울 25개구 내 탑승완료 건수: {len(seoul_completed):,}")
print(f"전체 탑승완료 대비 비율: {len(seoul_completed) / len(completed) * 100:.1f}%")
print(f"집계 출발구 수: {seoul_completed['출발구'].nunique():,}개")
print(f"집계 목적구 수: {seoul_completed['목적구'].nunique():,}개")


## 4. 같은 구 내 이동 추출

`출발구`와 `목적구`가 같은 행만 추출한다. 이 데이터는 서울 자치구 내부에서 발생한 탑승완료 이동을 의미한다.


In [ ]:
same_gu_seoul = seoul_completed[
    seoul_completed["출발구"] == seoul_completed["목적구"]
].copy()

diff_gu_seoul = seoul_completed[
    seoul_completed["출발구"] != seoul_completed["목적구"]
].copy()

print(f"서울→서울 탑승완료 건수: {len(seoul_completed):,}")
print(f"같은 구 내 이동 건수: {len(same_gu_seoul):,} ({len(same_gu_seoul) / len(seoul_completed) * 100:.1f}%)")
print(f"다른 구 간 이동 건수: {len(diff_gu_seoul):,} ({len(diff_gu_seoul) / len(seoul_completed) * 100:.1f}%)")
display(same_gu_seoul.head())


## 5. 같은 구 내 이동 건수 순위

자치구별 같은 구 내 이동 건수를 집계한다. 같은 구 내 이동 건수가 많은 지역은 자치구 내부에서 반복적으로 이동 수요가 발생하는 지역으로 볼 수 있다.


In [ ]:
same_gu_rank = (
    same_gu_seoul
    .groupby("출발구")
    .size()
    .reindex(SEOUL_25, fill_value=0)
    .rename("같은구_이동건수")
    .reset_index()
    .rename(columns={"index": "자치구"})
    .sort_values("같은구_이동건수", ascending=False)
    .reset_index(drop=True)
)

display(same_gu_rank)

plot_data = same_gu_rank.sort_values("같은구_이동건수", ascending=True)

fig, ax = plt.subplots(figsize=(10, 10))
ax.barh(plot_data["자치구"], plot_data["같은구_이동건수"], color="#4c78a8")
ax.set_title("서울 자치구별 같은 구 내 이동 건수")
ax.set_xlabel("탑승완료 건수")
ax.set_ylabel("자치구")
ax.grid(axis="x", alpha=0.3)

for index, value in enumerate(plot_data["같은구_이동건수"]):
    ax.text(value, index, f" {value:,.0f}", va="center")

plt.tight_layout()
plt.show()


## 6. 같은 구 내 이동 비율

각 자치구에서 출발한 서울 내 이동 중 같은 구 안에서 끝난 이동의 비율을 계산한다. 이 비율이 높으면 해당 자치구 내부 생활권 이동이 상대적으로 강하다고 볼 수 있다.


In [ ]:
origin_total = (
    seoul_completed
    .groupby("출발구")
    .size()
    .reindex(SEOUL_25, fill_value=0)
    .rename("서울내_출발건수")
)

same_gu_count = (
    same_gu_seoul
    .groupby("출발구")
    .size()
    .reindex(SEOUL_25, fill_value=0)
    .rename("같은구_이동건수")
)

same_gu_ratio = pd.concat([origin_total, same_gu_count], axis=1).reset_index()
same_gu_ratio = same_gu_ratio.rename(columns={"index": "자치구"})
same_gu_ratio["같은구_이동비율(%)"] = (
    same_gu_ratio["같은구_이동건수"] / same_gu_ratio["서울내_출발건수"] * 100
).round(2)

same_gu_ratio = same_gu_ratio.sort_values("같은구_이동비율(%)", ascending=False).reset_index(drop=True)

display(same_gu_ratio)

plot_data = same_gu_ratio.sort_values("같은구_이동비율(%)", ascending=True)

fig, ax = plt.subplots(figsize=(10, 10))
ax.barh(plot_data["자치구"], plot_data["같은구_이동비율(%)"], color="#72b7b2")
ax.set_title("서울 자치구별 같은 구 내 이동 비율")
ax.set_xlabel("같은 구 내 이동 비율(%)")
ax.set_ylabel("자치구")
ax.grid(axis="x", alpha=0.3)

for index, value in enumerate(plot_data["같은구_이동비율(%)"]):
    ax.text(value, index, f" {value:.1f}%", va="center")

plt.tight_layout()
plt.show()


## 7. 구별 승차거리 평균·중앙값

같은 구 내 이동이라도 자치구 면적과 생활권 구조에 따라 실제 이동거리는 다를 수 있다. 자치구별 승차거리 평균과 중앙값을 비교해 “같은 구 이동은 항상 짧다”는 단순 해석을 피한다.


In [ ]:
same_gu_distance_summary = (
    same_gu_seoul
    .assign(승차거리=pd.to_numeric(same_gu_seoul["승차거리"], errors="coerce"))
    .groupby("출발구")
    .agg(
        같은구_이동건수=("승차거리", "size"),
        평균승차거리=("승차거리", "mean"),
        중앙값승차거리=("승차거리", "median"),
        p90승차거리=("승차거리", lambda s: s.quantile(0.9)),
    )
    .reindex(SEOUL_25)
    .reset_index()
    .rename(columns={"출발구": "자치구"})
    .sort_values("중앙값승차거리", ascending=False)
    .reset_index(drop=True)
)

display(same_gu_distance_summary)

plot_data = same_gu_distance_summary.sort_values("중앙값승차거리", ascending=True)

fig, ax = plt.subplots(figsize=(10, 10))
ax.barh(plot_data["자치구"], plot_data["중앙값승차거리"], color="#59a14f")
ax.set_title("같은 구 내 이동의 자치구별 중앙값 승차거리")
ax.set_xlabel("중앙값 승차거리")
ax.set_ylabel("자치구")
ax.grid(axis="x", alpha=0.3)

for index, value in enumerate(plot_data["중앙값승차거리"]):
    if pd.notna(value):
        ax.text(value, index, f" {value:,.0f}", va="center")

plt.tight_layout()
plt.show()


## 8. 구별 대기시간 평균·중앙값

같은 구 내 이동의 대기시간을 `접수→배차`, `배차→승차`, `접수→승차`로 나누어 계산한다. 이를 통해 같은 구 이동이 많은 자치구에서 어떤 단계의 대기시간이 긴지 확인한다.


In [ ]:
same_gu_wait = same_gu_seoul.copy()

for col in ["접수일시", "배차일시", "승차일시"]:
    same_gu_wait[col] = pd.to_datetime(same_gu_wait[col], errors="coerce")

same_gu_wait["접수_배차_분"] = (
    same_gu_wait["배차일시"] - same_gu_wait["접수일시"]
).dt.total_seconds() / 60

same_gu_wait["배차_승차_분"] = (
    same_gu_wait["승차일시"] - same_gu_wait["배차일시"]
).dt.total_seconds() / 60

same_gu_wait["접수_승차_분"] = (
    same_gu_wait["승차일시"] - same_gu_wait["접수일시"]
).dt.total_seconds() / 60

wait_cols = ["접수_배차_분", "배차_승차_분", "접수_승차_분"]

invalid_wait_count = same_gu_wait[wait_cols].lt(0).any(axis=1).sum()
for col in wait_cols:
    same_gu_wait.loc[same_gu_wait[col] < 0, col] = pd.NA

print(f"음수 대기시간 행 수: {invalid_wait_count:,}건")

same_gu_wait_summary = (
    same_gu_wait
    .groupby("출발구")
    .agg(
        같은구_이동건수=("접수일시", "size"),
        평균_접수배차분=("접수_배차_분", "mean"),
        중앙값_접수배차분=("접수_배차_분", "median"),
        평균_배차승차분=("배차_승차_분", "mean"),
        중앙값_배차승차분=("배차_승차_분", "median"),
        평균_접수승차분=("접수_승차_분", "mean"),
        중앙값_접수승차분=("접수_승차_분", "median"),
    )
    .reindex(SEOUL_25)
    .reset_index()
    .rename(columns={"출발구": "자치구"})
    .sort_values("중앙값_접수승차분", ascending=False)
    .reset_index(drop=True)
)

display(same_gu_wait_summary)

plot_data = same_gu_wait_summary.sort_values("중앙값_접수승차분", ascending=True)

fig, ax = plt.subplots(figsize=(10, 10))
ax.barh(plot_data["자치구"], plot_data["중앙값_접수승차분"], color="#e45756")
ax.set_title("같은 구 내 이동의 자치구별 중앙값 접수→승차 시간")
ax.set_xlabel("중앙값 접수→승차 시간(분)")
ax.set_ylabel("자치구")
ax.grid(axis="x", alpha=0.3)

for index, value in enumerate(plot_data["중앙값_접수승차분"]):
    if pd.notna(value):
        ax.text(value, index, f" {value:,.1f}", va="center")

plt.tight_layout()
plt.show()


## 9. 같은 구 내 이동이 많은 구 해석

같은 구 내 이동 건수, 같은 구 내 이동 비율, 승차거리, 대기시간을 함께 묶어 해석한다. 이동 건수가 많고 비율도 높은 지역은 자치구 내부 생활권 수요가 강한 지역으로 볼 수 있으며, 여기에 대기시간이 길게 나타나면 내부 수요 대응의 병목 후보로 볼 수 있다.


In [ ]:
same_gu_analysis = (
    same_gu_rank
    .merge(same_gu_ratio, on=["자치구", "같은구_이동건수"], how="left")
    .merge(
        same_gu_distance_summary[["자치구", "평균승차거리", "중앙값승차거리"]],
        on="자치구",
        how="left",
    )
    .merge(
        same_gu_wait_summary[[
            "자치구",
            "평균_접수배차분",
            "중앙값_접수배차분",
            "평균_배차승차분",
            "중앙값_배차승차분",
            "평균_접수승차분",
            "중앙값_접수승차분",
        ]],
        on="자치구",
        how="left",
    )
    .sort_values("같은구_이동건수", ascending=False)
    .reset_index(drop=True)
)

display(same_gu_analysis)

print("같은 구 내 이동 건수 상위 5개구")
display(same_gu_analysis.head(5))

print("같은 구 내 이동 비율 상위 5개구")
display(same_gu_analysis.sort_values("같은구_이동비율(%)", ascending=False).head(5))

print("같은 구 내 이동 중 접수→승차 중앙값이 긴 5개구")
display(same_gu_analysis.sort_values("중앙값_접수승차분", ascending=False).head(5))
